# 图片爬取部分

## Wikipedia Commons

In [16]:
import httpx
from bs4 import BeautifulSoup
import re
import json
import pandas as pd
import os
import random
import time
from dotenv import load_dotenv
import sqlite3
load_dotenv()
current_dir = os.getcwd()

PROJECT_ROOT = os.path.dirname(os.path.dirname(current_dir))
db_path = os.path.join(PROJECT_ROOT, "data", "commons_image_manifest.sqlite")
PROJECT_ROOT

'/Users/yukun/projects/wakareeru'

###  Wikipedia commons标签处理

由于commons的标签全都由英文书写，而且对于包含片假名形式格式是按照罗马音处理的，但是又有些格式不统一，为了保证能正确搜索到Category，需要进行转换

In [17]:
OPERATOR_PREFIX = {
    "JR東日本": "JR East",
    "JR東海":   "JR Central",
    "JR西日本": "JR West",
    "JR九州":   "JR Kyushu",
    "JR北海道": "JR Hokkaido",
    "JR四国":   "JR Shikoku",
    "JR貨物":   "JR Freight",
}

_DIGRAPHS = {
    'キャ':'kya','キュ':'kyu','キョ':'kyo',
    'シャ':'sha','シュ':'shu','ショ':'sho',
    'チャ':'cha','チュ':'chu','チョ':'cho',
    'ニャ':'nya','ニュ':'nyu','ニョ':'nyo',
    'ヒャ':'hya','ヒュ':'hyu','ヒョ':'hyo',
    'ミャ':'mya','ミュ':'myu','ミョ':'myo',
    'リャ':'rya','リュ':'ryu','リョ':'ryo',
    'ギャ':'gya','ギュ':'gyu','ギョ':'gyo',
    'ジャ':'ja', 'ジュ':'ju', 'ジョ':'jo',
    'ビャ':'bya','ビュ':'byu','ビョ':'byo',
    'ピャ':'pya','ピュ':'pyu','ピョ':'pyo',
}
_SINGLE = {
    'ア':'a', 'イ':'i', 'ウ':'u', 'エ':'e', 'オ':'o',
    'カ':'ka','キ':'ki','ク':'ku','ケ':'ke','コ':'ko',
    'サ':'sa','シ':'shi','ス':'su','セ':'se','ソ':'so',
    'タ':'ta','チ':'chi','ツ':'tsu','テ':'te','ト':'to',
    'ナ':'na','ニ':'ni','ヌ':'nu','ネ':'ne','ノ':'no',
    'ハ':'ha','ヒ':'hi','フ':'fu','ヘ':'he','ホ':'ho',
    'マ':'ma','ミ':'mi','ム':'mu','メ':'me','モ':'mo',
    'ヤ':'ya','ユ':'yu','ヨ':'yo',
    'ラ':'ra','リ':'ri','ル':'ru','レ':'re','ロ':'ro',
    'ワ':'wa','ヲ':'wo','ン':'n',
    'ガ':'ga','ギ':'gi','グ':'gu','ゲ':'ge','ゴ':'go',
    'ザ':'za','ジ':'ji','ズ':'zu','ゼ':'ze','ゾ':'zo',
    'ダ':'da','デ':'de','ド':'do',
    'バ':'ba','ビ':'bi','ブ':'bu','ベ':'be','ボ':'bo',
    'パ':'pa','ピ':'pi','プ':'pu','ペ':'pe','ポ':'po',
}

# Commons 分类特例：key=(series, wiki_title 去掉 #anchor)，value=直接使用的搜索前缀
COMMONS_OVERRIDES: dict[tuple[str, str], str] = {
    ("481系", "国鉄485系電車"): "JNR 485",  # 481/483/485/489系同属485系 category
}


def parse_json_list(value: str) -> list[str]:
    return json.loads(value)


def _katakana_to_romaji(text: str) -> str:
    result, i = '', 0
    while i < len(text):
        two = text[i:i+2]
        if two in _DIGRAPHS:
            result += _DIGRAPHS[two]; i += 2
        elif text[i] in _SINGLE:
            result += _SINGLE[text[i]]; i += 1
        else:
            result += text[i]; i += 1
    return result


def _operator_prefixes(operator_jp: list[str], series_type: str, wiki_title: str) -> list[str]:
    if series_type == "新幹線電車":
        return ["Shinkansen"]
    if wiki_title.startswith("国鉄"):
        return ["JNR"]

    prefixes = []
    for op in operator_jp:
        prefix = OPERATOR_PREFIX.get(op, op)
        if prefix not in prefixes:
            prefixes.append(prefix)
    return prefixes


def series_to_commons_prefixes(series: str, operator_jp: list[str], series_type: str, wiki_title: str) -> list[str]:
    wiki_base = wiki_title.split("#", 1)[0]
    override = COMMONS_OVERRIDES.get((series, wiki_base))
    if override:
        return [override]

    name = re.sub(r'[系形]$', '', series)
    m = re.match(r'^([゠-ヿ]+)(.*)', name)

    prefixes = []
    # 蒸気機関車在 Commons 常用 "C57 steam locomotive"，而不是 "JNR C57"。
    if series_type == "蒸気機関車":
        prefixes.append(f"{name} steam locomotive")

    operator_prefixes = _operator_prefixes(operator_jp, series_type, wiki_title)
    for op in operator_prefixes:
        if m:
            romaji = _katakana_to_romaji(m.group(1)).capitalize()
            rest = m.group(2)
            candidates = [f"{op} {romaji} {rest}".strip()]
            if rest:
                candidates.append(f"{op} {rest}")
        else:
            candidates = [f"{op} {name}"]

        for prefix in candidates:
            if prefix not in prefixes:
                prefixes.append(prefix)

    return prefixes


COMMONS_API_URL = "https://commons.wikimedia.org/w/api.php"
COMMONS_HEADERS = {"User-Agent": "TrainDatasetBuilder/1.0 (research; fengyukunfyk@gmail.com)"}


def _commons_query(params: dict, max_retries: int = 3, base_sleep: float = 1.0) -> dict | None:
    for attempt in range(max_retries + 1):
        try:
            resp = httpx.get(COMMONS_API_URL, params=params, headers=COMMONS_HEADERS, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            if "error" in data:
                error = data["error"]
                raise RuntimeError(f'{error.get("code", "api-error")} {error.get("info", data)}')
            return data
        except (httpx.HTTPError, ValueError, RuntimeError) as exc:
            if attempt >= max_retries:
                print(f'⚠ Commons request failed: {params} ({exc})')
                return None
            sleep_s = base_sleep * (2 ** attempt) + random.uniform(0, 0.5)
            print(f'⚠ Commons temporary error, retrying in {sleep_s:.1f}s ({attempt + 1}/{max_retries})')
            time.sleep(sleep_s)


def fetch_commons_categories(prefix: str, limit: int = 20) -> list[str] | None:
    data = _commons_query({
        "action": "query",
        "list": "allcategories",
        "acprefix": prefix,
        "aclimit": limit,
        "format": "json",
    })
    if data is None:
        return None
    return [r["*"] for r in data.get("query", {}).get("allcategories", [])]


def fetch_parent_categories(category: str) -> list[str] | None:
    # Commons category 是图结构；parent 只用来确认 exact 是否应提升到 series root。
    params = {
        "action": "query",
        "titles": f"Category:{category}",
        "prop": "categories",
        "cllimit": "max",
        "format": "json",
    }
    parents = []
    while True:
        data = _commons_query(params)
        if data is None:
            return None
        for page in data.get("query", {}).get("pages", {}).values():
            for cat in page.get("categories", []):
                parents.append(cat["title"].removeprefix("Category:"))
        if "continue" not in data:
            return parents
        params.update(data["continue"])


def fetch_subcategories(category: str) -> list[str] | None:
    # 这里只取直接子 category，不取文件；后续 flatten 阶段再决定递归深度。
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": f"Category:{category}",
        "cmtype": "subcat",
        "cmlimit": "max",
        "format": "json",
    }
    subcats = []
    while True:
        data = _commons_query(params)
        if data is None:
            return None
        for item in data.get("query", {}).get("categorymembers", []):
            subcats.append(item["title"].removeprefix("Category:"))
        if "continue" not in data:
            return subcats
        params.update(data["continue"])


def search_commons_category(
    series: str, operator_jp: list[str], series_type: str, wiki_title: str
) -> tuple[str, list[str]]:
    prefixes = series_to_commons_prefixes(series, operator_jp, series_type, wiki_title)
    for prefix in prefixes:
        cats = fetch_commons_categories(prefix)
        if cats is None:
            continue
        if cats:
            return prefix, cats
    return prefixes[0], []





### Commons root category 选择

先只找到每条车型记录对应的主 category。大部分使用 prefix 完全匹配；如果同时存在 `prefix` 和 `prefix series`，再用父子关系保守判断是否提升到 `series`。空结果和弱匹配留给人工 override。

In [18]:
# Root category 特例：key=(series, wiki_title 去掉 #anchor)，value=人工确认的 Commons 主分类
COMMONS_ROOT_OVERRIDES: dict[tuple[str, str], str] = {}


def _wiki_base(wiki_title: str) -> str:
    return wiki_title.split("#", 1)[0]


def _dedupe(items: list[str]) -> list[str]:
    out = []
    for item in items:
        if item not in out:
            out.append(item)
    return out


def _promote_to_series(series_label: str, exact: str, series_cat: str) -> tuple[bool, str]:
    # 只有日文 label 是 系，且 Commons 图确认二者父子关系时才提升到 ... series。
    if not series_label.endswith("系"):
        return False, "exact match; row is not 系"

    exact_parents = fetch_parent_categories(exact) or []
    if series_cat in exact_parents:
        return True, f'promoted: "{series_cat}" is parent of "{exact}"'

    series_children = fetch_subcategories(series_cat) or []
    if exact in series_children:
        return True, f'promoted: "{exact}" is child of "{series_cat}"'

    return False, "exact match; series relation not confirmed"


def choose_commons_root(series_label: str, prefix: str, candidates: list[str]) -> dict:
    # 大多数车型直接用 prefix exact match；蒸気機関車等复数主类要避开单车 category。
    exact = prefix if prefix in candidates else None
    series_cat = f"{prefix} series" if f"{prefix} series" in candidates else None
    plural = f"{prefix}s" if f"{prefix}s" in candidates else None

    if plural:
        return {"root": plural, "decision": "plural category match", "needs_review": False}

    if exact and series_cat:
        promote, reason = _promote_to_series(series_label, exact, series_cat)
        root = series_cat if promote else exact
        return {"root": root, "decision": reason, "needs_review": False}

    if exact:
        return {"root": exact, "decision": "exact prefix match", "needs_review": False}

    if series_cat:
        return {"root": series_cat, "decision": "series category match without exact", "needs_review": False}

    if candidates:
        return {"root": candidates[0], "decision": "fallback to first prefix result", "needs_review": True}

    return {"root": None, "decision": "no Commons category candidates", "needs_review": True}


def _find_root_from_prefixes(series_label: str, prefixes: list[str], category_limit: int = 20) -> dict:
    all_candidates = []
    fallback = None

    for prefix in prefixes:
        candidates = fetch_commons_categories(prefix, limit=category_limit)
        if candidates is None:
            continue
        all_candidates.extend(candidates)
        if not candidates:
            continue

        chosen = choose_commons_root(series_label, prefix, candidates)
        result = {
            "commons_prefix": prefix,
            "commons_root_category": chosen["root"],
            "commons_root_decision": chosen["decision"],
            "commons_candidates": _dedupe(all_candidates),
            "needs_review": chosen["needs_review"],
        }
        if not chosen["needs_review"]:
            return result
        # 弱匹配先暂存，继续尝试后续 prefix，避免错过更好的 exact match。
        fallback = fallback or result

    if fallback:
        fallback["commons_candidates"] = _dedupe(all_candidates)
        return fallback

    return {
        "commons_prefix": prefixes[0],
        "commons_root_category": None,
        "commons_root_decision": "no prefix returned categories",
        "commons_candidates": _dedupe(all_candidates),
        "needs_review": True,
    }


def find_commons_root(row: pd.Series, category_limit: int = 20) -> dict:
    override = COMMONS_ROOT_OVERRIDES.get((row["series"], _wiki_base(row["wiki_title"])))
    prefixes = series_to_commons_prefixes(row["series"], row["operator_jp"], row["type"], row["wiki_title"])

    if override:
        return {
            "commons_prefix": prefixes[0],
            "commons_root_category": override,
            "commons_root_decision": "manual override",
            "commons_operator_roots": {op: override for op in row["operator_en"]},
            "commons_candidates": [],
            "needs_review": False,
        }

    root = _find_root_from_prefixes(row["series"], prefixes, category_limit=category_limit)
    operator_roots = {}
    for op in row["operator_en"]:
        op_prefixes = series_to_commons_prefixes(row["series"], [op], row["type"], row["wiki_title"])
        op_root = _find_root_from_prefixes(row["series"], op_prefixes, category_limit=category_limit)
        if op_root["commons_root_category"]:
            operator_roots[op] = op_root["commons_root_category"]

    root["commons_operator_roots"] = operator_roots
    return root


In [19]:
all_model = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "jr_east_series.csv"))
for col in ["operator_page_title", "operator_jp", "operator_en"]:
    all_model[col] = all_model[col].apply(parse_json_list)

In [20]:
root_rows = []

for _, row in all_model.iterrows():
    result = find_commons_root(row)
    root_rows.append(result)
    status = "?" if result["needs_review"] else "✓"
    print(
        f'{status} {row["series"]} ({", ".join(row["operator_jp"])}) '
        f'→ "{result["commons_root_category"]}" [{result["commons_root_decision"]}]'
    )

commons_root_df = pd.DataFrame(root_rows)
for col in commons_root_df.columns:
    all_model[col] = commons_root_df[col]

print(f'\n共 {len(all_model)} 条，需要人工确认 {all_model["needs_review"].sum()} 条')
all_model[[
    "series", "operator_jp", "commons_prefix", "commons_root_category",
    "commons_operator_roots", "commons_root_decision", "needs_review", "commons_candidates"
]]


✓ E2系 (JR東日本) → "Shinkansen E2" [exact prefix match]
✓ E3系 (JR東日本) → "Shinkansen E3" [exact prefix match]
✓ E5系 (JR東日本) → "Shinkansen E5" [exact prefix match]
✓ E6系 (JR東日本) → "Shinkansen E6" [exact prefix match]
✓ E7系 (JR東日本) → "Shinkansen E7" [exact prefix match]
✓ E8系 (JR東日本) → "Shinkansen E8" [exact prefix match]
✓ E926形 (JR東日本) → "Shinkansen E926" [exact prefix match]
✓ E956形 (JR東日本) → "Shinkansen E956" [exact prefix match]
✓ C57形 (JR東日本) → "C57 steam locomotives" [plural category match]
✓ C58形 (JR東日本) → "C58 steam locomotives" [plural category match]
✓ C61形 (JR東日本) → "C61 steam locomotives" [plural category match]
✓ D51形 (JR東日本) → "D51 steam locomotives" [plural category match]
✓ EF64形 (JR東日本) → "JNR EF64" [exact prefix match]
✓ EF65形 (JR東日本) → "JNR EF65" [exact prefix match]
✓ EF81形 (JR東日本) → "JNR EF81" [exact prefix match]
✓ DD14形 (JR東日本) → "JNR DD14" [exact prefix match]
✓ DD51形 (JR東日本) → "JNR DD51" [exact prefix match]
✓ DE10形 (JR東日本) → "JNR DE10" [exact prefix match]
✓ 157系 (

,series,operator_jp,commons_prefix,commons_root_category,commons_operator_roots,commons_root_decision,needs_review,commons_candidates
0,E2系,[JR東日本],Shinkansen E2,Shinkansen E2,{'JR East': 'Shinkansen E2'},exact prefix match,False,"[Shinkansen E2, Shinkansen E2 (200 series live..."
1,E3系,[JR東日本],Shinkansen E3,Shinkansen E3,{'JR East': 'Shinkansen E3'},exact prefix match,False,"[Shinkansen E3, Shinkansen E3-0, Shinkansen E3..."
2,E5系,[JR東日本],Shinkansen E5,Shinkansen E5,{'JR East': 'Shinkansen E5'},exact prefix match,False,"[Shinkansen E5, Shinkansen E5 at Hachinohe Sta..."
3,E6系,[JR東日本],Shinkansen E6,Shinkansen E6,{'JR East': 'Shinkansen E6'},exact prefix match,False,"[Shinkansen E6, Shinkansen E6 at Akita Station..."
4,E7系,[JR東日本],Shinkansen E7,Shinkansen E7,{'JR East': 'Shinkansen E7'},exact prefix match,False,"[Shinkansen E7, Shinkansen E7 (pink line liver..."
...,...,...,...,...,...,...,...,...
141,キハ38形,[JR東日本],JNR Kiha 38,JNR Kiha 38,{'JR East': 'JNR Kiha 38'},exact prefix match,False,[JNR Kiha 38]
142,キハ45系,[JR東日本],JNR Kiha 45,JNR Kiha 45 series,{'JR East': 'JNR Kiha 45 series'},series category match without exact,False,"[JNR Kiha 45 (JR Shikoku), JNR Kiha 45 (JR Wes..."
143,キハ141系,[JR東日本],JR East Kiha 141,NaN,{},no prefix returned categories,True,[]
144,キヤ191系,[JR東日本],JNR Kiya 191,JNR Kiya 191,{'JR East': 'JNR Kiya 191'},exact prefix match,False,[JNR Kiya 191]


In [21]:
all_model

,series,wiki_title,status,type,subtype,operator_page_title,operator_jp,operator_en,full_name,commons_prefix,commons_root_category,commons_root_decision,commons_candidates,needs_review,commons_operator_roots
0,E2系,新幹線E2系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E2系電車,Shinkansen E2,Shinkansen E2,exact prefix match,"[Shinkansen E2, Shinkansen E2 (200 series live...",False,{'JR East': 'Shinkansen E2'}
1,E3系,新幹線E3系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E3系電車,Shinkansen E3,Shinkansen E3,exact prefix match,"[Shinkansen E3, Shinkansen E3-0, Shinkansen E3...",False,{'JR East': 'Shinkansen E3'}
2,E5系,新幹線E5系・H5系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E5系・H5系電車,Shinkansen E5,Shinkansen E5,exact prefix match,"[Shinkansen E5, Shinkansen E5 at Hachinohe Sta...",False,{'JR East': 'Shinkansen E5'}
3,E6系,新幹線E6系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E6系電車,Shinkansen E6,Shinkansen E6,exact prefix match,"[Shinkansen E6, Shinkansen E6 at Akita Station...",False,{'JR East': 'Shinkansen E6'}
4,E7系,新幹線E7系・W7系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E7系・W7系電車,Shinkansen E7,Shinkansen E7,exact prefix match,"[Shinkansen E7, Shinkansen E7 (pink line liver...",False,{'JR East': 'Shinkansen E7'}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,キハ38形,国鉄キハ38形気動車,廃止,気動車,一般形,[JR東日本の車両形式],[JR東日本],[JR East],国鉄キハ38形気動車,JNR Kiha 38,JNR Kiha 38,exact prefix match,[JNR Kiha 38],False,{'JR East': 'JNR Kiha 38'}
142,キハ45系,国鉄キハ45系気動車,廃止,気動車,一般形,[JR東日本の車両形式],[JR東日本],[JR East],国鉄キハ45系気動車,JNR Kiha 45,JNR Kiha 45 series,series category match without exact,"[JNR Kiha 45 (JR Shikoku), JNR Kiha 45 (JR Wes...",False,{'JR East': 'JNR Kiha 45 series'}
143,キハ141系,JR北海道キハ141系気動車,廃止,気動車,一般形,[JR東日本の車両形式],[JR東日本],[JR East],JR北海道キハ141系気動車,JR East Kiha 141,NaN,no prefix returned categories,[],True,{}
144,キヤ191系,国鉄キヤ191系気動車,廃止,気動車,事業用,[JR東日本の車両形式],[JR東日本],[JR East],国鉄キヤ191系気動車,JNR Kiya 191,JNR Kiya 191,exact prefix match,[JNR Kiya 191],False,{'JR East': 'JNR Kiya 191'}


In [22]:
print(all_model[all_model['series'] == 'EF510形']['commons_operator_roots'])

79    {'JR East': 'JR East EF510-500'}
Name: commons_operator_roots, dtype: object


In [23]:
def set_model_values(series: str, **values) -> None:
    idx = all_model.index[all_model["series"] == series]
    for i in idx:
        for col, value in values.items():
            all_model.at[i, col] = value


# 修正 JR 北海道的车所属问题；保持 operator 字段为 list，方便后续多 operator 合并。
set_model_values(
    "キハ141系",
    operator_jp=["JR北海道"],
    operator_en=["JR Hokkaido"],
)

# 修正 921形 到 922形。
all_model.loc[all_model["series"] == "921形", "series"] = "922形"
set_model_values(
    "922形",
    operator_jp=["国鉄"],
    operator_en=["JNR"],
    commons_prefix="Shinkansen 922",
    commons_root_category="Shinkansen 922",
    commons_root_decision="manual correction",
    commons_operator_roots={"JNR": "Shinkansen 922"},
    needs_review=False,
)

# 合并 952形 和 953形。
all_model.loc[all_model["series"] == "952形", "series"] = "952/953形"
all_model.drop(all_model[all_model["series"] == "953形"].index, inplace=True)

# EF510 同时有 JR 東日本 500番台和 JR 貨物主类；单值 root 放车型总体，operator_roots 保留细分入口。
set_model_values(
    "EF510形",
    operator_jp=["JR東日本", "JR貨物"],
    operator_en=["JR East", "JR Freight"],
    commons_prefix="JR Freight EF510",
    commons_root_category="JR Freight EF510",
    commons_root_decision="manual multi-operator root",
    commons_operator_roots={
        "JR East": "JR East EF510-500",
        "JR Freight": "JR Freight EF510",
    },
    needs_review=False,
)


### 开始手动补充

In [24]:
all_model[all_model['needs_review'] == True][['series', 'wiki_title', 'commons_root_category', 'commons_root_decision', 'commons_candidates', 'operator_jp', 'operator_en']]

,series,wiki_title,commons_root_category,commons_root_decision,commons_candidates,operator_jp,operator_en
51,キハ100系,JR東日本キハ100系・キハ110系気動車,NaN,no prefix returned categories,[],[JR東日本],[JR East]
52,キハ110系,JR東日本キハ100系・キハ110系気動車,NaN,no prefix returned categories,[],[JR東日本],[JR East]
70,952/953形,新幹線952形・953形電車,Shinkansen 952/953,fallback to first prefix result,[Shinkansen 952/953],[JR東日本],[JR East]
86,DD17形,国鉄DD17形ディーゼル機関車,NaN,no prefix returned categories,[],[JR東日本],[JR East]
87,DD18形,JR東日本DD18形ディーゼル機関車,NaN,no prefix returned categories,[],[JR東日本],[JR East]
88,DD19形,国鉄DD17形ディーゼル機関車#DD19形,NaN,no prefix returned categories,[],[JR東日本],[JR East]
98,483系,国鉄485系電車,NaN,no prefix returned categories,[],[JR東日本],[JR East]
105,453系,国鉄457系電車,NaN,no prefix returned categories,[],[JR東日本],[JR East]
127,901系,JR東日本209系電車,NaN,no prefix returned categories,[],[JR東日本],[JR East]
135,クモヤ743形,JR東日本クモヤ743形電車,NaN,no prefix returned categories,[],[JR東日本],[JR East]


In [25]:
completion = {
    "キハ100系": "JR East KiHa 100 series",
    "キハ110系": "JR East KiHa 110 series",
    "DD19形": "JR East DD19",
    "キハ141系": "JR Hokkaido Kiha 141",
}


def _operator_root_map(operator_en: list[str], commons_root: str) -> dict[str, str]:
    return {operator: commons_root for operator in operator_en}


for series, commons_root in completion.items():
    idx = all_model.index[all_model["series"] == series]
    for i in idx:
        all_model.at[i, "commons_prefix"] = commons_root
        all_model.at[i, "commons_root_category"] = commons_root
        all_model.at[i, "commons_root_decision"] = "manual completion"
        all_model.at[i, "commons_operator_roots"] = _operator_root_map(all_model.at[i, "operator_en"], commons_root)
        all_model.at[i, "needs_review"] = False

all_model[all_model["series"].isin(completion.keys())][[
    "series", "operator_en", "commons_root_category", "commons_operator_roots", "needs_review"
]]


,series,operator_en,commons_root_category,commons_operator_roots,needs_review
51,キハ100系,[JR East],JR East KiHa 100 series,{'JR East': 'JR East KiHa 100 series'},False
52,キハ110系,[JR East],JR East KiHa 110 series,{'JR East': 'JR East KiHa 110 series'},False
88,DD19形,[JR East],JR East DD19,{'JR East': 'JR East DD19'},False
143,キハ141系,[JR Hokkaido],JR Hokkaido Kiha 141,{'JR Hokkaido': 'JR Hokkaido Kiha 141'},False


In [26]:
POWER_TYPE_MAP_EXPORT = {
    "電車": "EMU",
    "新幹線電車": "EMU",
    "気動車": "DMU",
    "電気機関車": "Electric Locomotive",
    "ディーゼル機関車": "Diesel Locomotive",
    "蒸気機関車": "Steam Locomotive",
    "電気・ディーゼル両用（EDC方式）車両": "Electro-diesel Multiple Unit",
}

all_model["power_type"] = all_model["type"].map(POWER_TYPE_MAP_EXPORT)
display(all_model[all_model["power_type"].isna()][["series", "wiki_title", "type"]])
all_model.to_csv(os.path.join(PROJECT_ROOT, "data", "jr_east_freight_series_wiki_commons.csv"), index=False)


,series,wiki_title,type


## 图片抓取

先抽取小部分样本作为实验，从根目录逐渐到递归爬取，并标注metadata存入manifest，同时需要一个sqlite数据库来处理续传，此外，爬取时分别对文件名和subcategory做excluding，例如文件名遇到seat, interior直接排除，subcategory遇到interior直接排除

### Manifest 数据库

第一步只从 `commons_root_category` 抓 root category 里的文件，写入 SQLite manifest；暂不递归 subcategory，也不下载原图。后续递归、operator scope、下载状态都留字段扩展。

In [ ]:
FILE_EXCLUDE_PATTERNS = (
    "interior", "inside", "seat", "seats", "seating", "reclining", "free-space",
    "cab", "cockpit", "toilet", "wc", "route map", "counter", "merchandising counter",
    "display", "lcd", "vvvf", "logo", "air cleaner", "antenna", "pantograph",
    "camera", "accident", "syanai", "車内", "運転台", "運転室", "トイレ", "便所","カメラ", "事故", "車内",
    "trainchannel",
    "運転台", "運転室", "トイレ", "便所",
    "洗面所", "洗面台", "モニター", "カウンター", "停車駅案内", "案内表示器",
    "パンタグラフ", "エアクリーナー", "集電装置", "エアコン", "クーラー",
)

CATEGORY_EXCLUDE_PATTERNS = (
    "interior", "inside", "parts", "seats", "information display", "mockup","green car"
)

# Recursive Commons categories sometimes drift into through-service/private-railway cars.
# Keep these series-scoped so future Tokyu/Toei labels are not filtered globally.
SERIES_CATEGORY_EXCLUDE_PATTERNS = {
    #"E231系": ("tokyu", "tōkyū", "toei", "shibuya hikarie"),
}


In [ ]:
import ast
import sqlite3
from datetime import datetime, timezone

COMMONS_MODEL_CSV = os.path.join(PROJECT_ROOT, "data", "jr_east_freight_series_wiki_commons.csv")
IMAGE_DB_PATH = os.path.join(PROJECT_ROOT, "data", "commons_image_manifest.sqlite")

POWER_TYPE_MAP = {
    "電車": "EMU",
    "新幹線電車": "EMU",
    "気動車": "DMU",
    "電気機関車": "Electric Locomotive",
    "ディーゼル機関車": "Diesel Locomotive",
    "蒸気機関車": "Steam Locomotive",
    "電気・ディーゼル両用（EDC方式）車両": "Electro-diesel Multiple Unit",
}


def utc_now() -> str:
    """Return an ISO timestamp for manifest writes."""
    return datetime.now(timezone.utc).isoformat()


def parse_literal(value, default):
    """Parse list/dict values exported by pandas CSV round-trips."""
    if isinstance(value, (list, dict)):
        return value
    if pd.isna(value) or value == "":
        return default
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return default


def map_power_type(type_value: str | None) -> str | None:
    """Map the Japanese rolling-stock type into the English power type taxonomy."""
    if pd.isna(type_value):
        return None
    return POWER_TYPE_MAP.get(str(type_value).strip())


def load_commons_models(path: str = COMMONS_MODEL_CSV) -> pd.DataFrame:
    """Load model rows with Commons root metadata from CSV and mapped power_type."""
    df = pd.read_csv(path)
    for col in ["operator_page_title", "operator_jp", "operator_en", "commons_candidates"]:
        if col in df:
            df[col] = df[col].apply(lambda v: parse_literal(v, []))
    if "commons_operator_roots" in df:
        df["commons_operator_roots"] = df["commons_operator_roots"].apply(lambda v: parse_literal(v, {}))
    if "type" in df:
        df["power_type"] = df["type"].apply(map_power_type)
    return df


In [27]:
def init_image_db(db_path: str = IMAGE_DB_PATH) -> sqlite3.Connection:
    """Open the image manifest database and create/migrate required tables."""
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS categories (
            category TEXT PRIMARY KEY,
            parent_category TEXT,
            source_scope TEXT NOT NULL DEFAULT 'root',
            fetched_at TEXT,
            fetch_status TEXT NOT NULL DEFAULT 'pending',
            error TEXT
        );

        CREATE TABLE IF NOT EXISTS images (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            series TEXT NOT NULL,
            wiki_title TEXT,
            power_type TEXT,
            operator_en_json TEXT NOT NULL,
            root_category TEXT NOT NULL,
            category TEXT NOT NULL,
            category_path_json TEXT NOT NULL,
            file_title TEXT NOT NULL,
            pageid INTEGER,
            image_url TEXT,
            thumb_url TEXT,
            mime TEXT,
            width INTEGER,
            height INTEGER,
            size INTEGER,
            sha1 TEXT,
            extmetadata_json TEXT,
            excluded INTEGER NOT NULL DEFAULT 0,
            exclude_reason TEXT,
            download_status TEXT NOT NULL DEFAULT 'not_started',
            downloaded_path TEXT,
            fetched_at TEXT NOT NULL,
            UNIQUE(series, category, file_title)
        );

        CREATE TABLE IF NOT EXISTS image_categories (
            file_title TEXT NOT NULL,
            category TEXT NOT NULL,
            source_scope TEXT NOT NULL DEFAULT 'root',
            PRIMARY KEY (file_title, category)
        );
        """
    )

    # Existing local DBs created before category_path_json need a one-column migration.
    image_columns = {row[1] for row in conn.execute("PRAGMA table_info(images)")}
    if "category_path_json" not in image_columns:
        conn.execute("ALTER TABLE images ADD COLUMN category_path_json TEXT NOT NULL DEFAULT '[]'")
    if "power_type" not in image_columns:
        conn.execute("ALTER TABLE images ADD COLUMN power_type TEXT")

    category_columns = {row[1] for row in conn.execute("PRAGMA table_info(categories)")}
    if "parent_category" not in category_columns:
        conn.execute("ALTER TABLE categories ADD COLUMN parent_category TEXT")

    conn.commit()
    return conn


In [9]:
conn = init_image_db()
conn.close()
IMAGE_DB_PATH


'/Users/yukun/projects/wakareeru/data/commons_image_manifest.sqlite'

In [ ]:
def repair_image_power_type_from_series(
    db_path: str = IMAGE_DB_PATH,
    model_csv: str = COMMONS_MODEL_CSV,
) -> pd.DataFrame:
    """One-off repair: backfill images.power_type from the model CSV by series."""
    models = load_commons_models(model_csv)
    power_by_series = (
        models[["series", "power_type"]]
        .dropna(subset=["power_type"])
        .drop_duplicates(subset=["series"], keep="last")
    )
    update_rows = power_by_series[["power_type", "series"]].values.tolist()

    conn = init_image_db(db_path)
    try:
        before_missing = conn.execute(
            "SELECT COUNT(*) FROM images WHERE power_type IS NULL OR power_type = ''"
        ).fetchone()[0]
        conn.executemany(
            """
            UPDATE images
            SET power_type = ?
            WHERE series = ?
              AND (power_type IS NULL OR power_type = '')
            """,
            update_rows,
        )
        conn.commit()
        after_missing = conn.execute(
            "SELECT COUNT(*) FROM images WHERE power_type IS NULL OR power_type = ''"
        ).fetchone()[0]

        print(f"series mappings available: {len(update_rows)}")
        print(f"missing power_type before: {before_missing}")
        print(f"missing power_type after: {after_missing}")
        return pd.read_sql_query(
            """
            SELECT power_type, COUNT(*) AS n
            FROM images
            GROUP BY power_type
            ORDER BY n DESC
            """,
            conn,
        )
    finally:
        conn.close()


# Run this once after updating the schema helpers to backfill the existing DB.
# power_type_repair_summary = repair_image_power_type_from_series()
# display(power_type_repair_summary)


series mappings available: 145
missing power_type before: 2363
missing power_type after: 0


,power_type,n
0,DMU,1038
1,EMU,935
2,Steam Locomotive,251
3,Diesel Locomotive,105
4,Electric Locomotive,34


In [ ]:
def has_excluded_pattern(text: str, patterns: tuple[str, ...]) -> str | None:
    """Return the first exclusion pattern found in text."""
    lowered = (text or "").lower()
    for pattern in patterns:
        if pattern.lower() in lowered:
            return pattern
    return None


def category_exclude_reason(row: pd.Series, category: str) -> str | None:
    """Return a category-level exclusion reason for this series/category pair."""
    category_exclude = has_excluded_pattern(category, CATEGORY_EXCLUDE_PATTERNS)
    if category_exclude:
        return f"category:{category_exclude}"

    series_patterns = SERIES_CATEGORY_EXCLUDE_PATTERNS.get(row["series"], ())
    series_exclude = has_excluded_pattern(category, series_patterns)
    if series_exclude:
        return f"category:wrong-series:{series_exclude}"

    return None


def fetch_category_file_members(category: str, max_files: int = 10) -> list[dict]:
    """Fetch direct file members of a Commons category."""
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": f"Category:{category}",
        "cmtype": "file",
        "cmlimit": min(max_files, 50),
        "format": "json",
    }
    files = []
    while len(files) < max_files:
        data = _commons_query(params)
        if data is None:
            break
        files.extend(data.get("query", {}).get("categorymembers", []))
        if "continue" not in data or len(files) >= max_files:
            break
        params.update(data["continue"])
        params["cmlimit"] = min(max_files - len(files), 50)
    return files[:max_files]


def fetch_imageinfo(file_titles: list[str], thumb_width: int = 512) -> dict[str, dict]:
    """Fetch image URLs, dimensions, mime, sha1, and Commons metadata."""
    result = {}
    for start in range(0, len(file_titles), 50):
        batch = file_titles[start:start + 50]
        data = _commons_query({
            "action": "query",
            "titles": "|".join(batch),
            "prop": "imageinfo",
            "iiprop": "url|mime|size|sha1|extmetadata",
            "iiurlwidth": thumb_width,
            "format": "json",
        })
        if data is None:
            continue
        for page in data.get("query", {}).get("pages", {}).values():
            title = page.get("title")
            info = (page.get("imageinfo") or [{}])[0]
            result[title] = {"pageid": page.get("pageid"), **info}
    return result


def build_image_records(
    row: pd.Series, category: str, max_files: int = 10, category_path: list[str] | None = None
) -> list[dict]:
    """Build manifest records for files directly under one category."""
    category_path = category_path or [category]
    category_exclude = category_exclude_reason(row, category)
    members = fetch_category_file_members(category, max_files=max_files)
    titles = [m["title"] for m in members]
    info_by_title = fetch_imageinfo(titles)

    records = []
    for member in members:
        file_title = member["title"]
        info = info_by_title.get(file_title, {})
        file_exclude = has_excluded_pattern(file_title, FILE_EXCLUDE_PATTERNS)
        exclude_reason = None
        if category_exclude:
            exclude_reason = category_exclude
        elif file_exclude:
            exclude_reason = f"file:{file_exclude}"

        records.append({
            "series": row["series"],
            "wiki_title": row.get("wiki_title"),
            "power_type": None if pd.isna(row.get("power_type")) else row.get("power_type"),
            "operator_en_json": json.dumps(row.get("operator_en", []), ensure_ascii=False),
            "root_category": row["commons_root_category"],
            "category": category,
            "category_path_json": json.dumps(category_path, ensure_ascii=False),
            "file_title": file_title,
            "pageid": info.get("pageid") or member.get("pageid"),
            "image_url": info.get("url"),
            "thumb_url": info.get("thumburl"),
            "mime": info.get("mime"),
            "width": info.get("width"),
            "height": info.get("height"),
            "size": info.get("size"),
            "sha1": info.get("sha1"),
            "extmetadata_json": json.dumps(info.get("extmetadata", {}), ensure_ascii=False),
            "excluded": int(exclude_reason is not None),
            "exclude_reason": exclude_reason,
            "fetched_at": utc_now(),
        })
    return records


### MIME 过滤

Commons 的 `Category` 里会混入 PDF、音频、视频等非图片文件。这里不在爬取阶段临时过滤，而是在 manifest 写入 SQLite 后，直接对数据库执行 MIME 清理：只保留 `images.mime` 以 `image/` 开头的记录，并同步清理 `image_categories` 关联表。

In [ ]:
def purge_non_image_manifest_records(conn: sqlite3.Connection) -> int:
    """Delete existing DB rows whose MIME is missing or not an image/* type."""
    conn.execute(
        """
        DELETE FROM image_categories
        WHERE EXISTS (
            SELECT 1
            FROM images
            WHERE images.file_title = image_categories.file_title
              AND images.category = image_categories.category
              AND LOWER(COALESCE(images.mime, '')) NOT LIKE 'image/%'
        )
        """
    )
    deleted = conn.execute(
        "DELETE FROM images WHERE LOWER(COALESCE(mime, '')) NOT LIKE 'image/%'"
    ).rowcount
    conn.commit()
    return deleted


def apply_mime_filter_to_manifest_db(db_path: str = IMAGE_DB_PATH) -> dict[str, int]:
    """Apply the image-only MIME filter directly to the manifest database."""
    conn = init_image_db(db_path)
    try:
        before = conn.execute("SELECT COUNT(*) FROM images").fetchone()[0]
        non_image = conn.execute(
            "SELECT COUNT(*) FROM images WHERE LOWER(COALESCE(mime, '')) NOT LIKE 'image/%'"
        ).fetchone()[0]
        deleted = purge_non_image_manifest_records(conn)
        after = conn.execute("SELECT COUNT(*) FROM images").fetchone()[0]
        return {"before": before, "non_image": non_image, "deleted": deleted, "after": after}
    finally:
        conn.close()


def upsert_image_records(
    conn: sqlite3.Connection,
    category: str,
    records: list[dict],
    source_scope: str = "root",
    parent_category: str | None = None,
) -> None:
    """Upsert image records while preserving future download status fields."""
    conn.execute(
        """
        INSERT INTO categories(category, parent_category, source_scope, fetched_at, fetch_status, error)
        VALUES (?, ?, ?, ?, 'ok', NULL)
        ON CONFLICT(category) DO UPDATE SET
            parent_category=COALESCE(excluded.parent_category, categories.parent_category),
            source_scope=excluded.source_scope,
            fetched_at=excluded.fetched_at,
            fetch_status='ok',
            error=NULL
        """,
        (category, parent_category, source_scope, utc_now()),
    )
    for record in records:
        conn.execute(
            """
            INSERT INTO images(
                series, wiki_title, power_type, operator_en_json, root_category, category, category_path_json, file_title,
                pageid, image_url, thumb_url, mime, width, height, size, sha1, extmetadata_json,
                excluded, exclude_reason, fetched_at
            ) VALUES (
                :series, :wiki_title, :power_type, :operator_en_json, :root_category, :category, :category_path_json, :file_title,
                :pageid, :image_url, :thumb_url, :mime, :width, :height, :size, :sha1, :extmetadata_json,
                :excluded, :exclude_reason, :fetched_at
            )
            ON CONFLICT(series, category, file_title) DO UPDATE SET
                pageid=excluded.pageid,
                image_url=excluded.image_url,
                thumb_url=excluded.thumb_url,
                power_type=excluded.power_type,
                mime=excluded.mime,
                width=excluded.width,
                height=excluded.height,
                size=excluded.size,
                sha1=excluded.sha1,
                category_path_json=excluded.category_path_json,
                extmetadata_json=excluded.extmetadata_json,
                excluded=excluded.excluded,
                exclude_reason=excluded.exclude_reason,
                fetched_at=excluded.fetched_at
            """,
            record,
        )
        conn.execute(
            """
            INSERT OR REPLACE INTO image_categories(file_title, category, source_scope)
            VALUES (?, ?, ?)
            """,
            (record["file_title"], category, source_scope),
        )
    conn.commit()


In [11]:
# 清理已有数据库中的非图片记录，避免后续处理时遇到不符合预期的 MIME 类型。
filter_result = apply_mime_filter_to_manifest_db()
filter_result

{'before': 2363, 'non_image': 0, 'deleted': 0, 'after': 2363}

In [12]:
def should_skip_category(row: pd.Series, category: str) -> str | None:
    """Return the exclusion reason for a category, if it should not be crawled."""
    return category_exclude_reason(row, category)


def collect_category_records(
    row: pd.Series,
    category: str,
    path: list[str],
    depth: int,
    max_depth: int,
    max_files_per_category: int,
    visited_paths: set[tuple[str, ...]],
) -> list[dict]:
    """Collect image records from a category and optionally recurse into subcategories."""
    path_key = tuple(path)
    if path_key in visited_paths:
        return []
    visited_paths.add(path_key)

    records = build_image_records(
        row, category, max_files=max_files_per_category, category_path=path
    )
    print(f'depth={depth} Category:{category} -> {len(records)} files')

    if depth >= max_depth:
        return records

    subcats = fetch_subcategories(category) or []
    for subcat in subcats:
        if should_skip_category(row, subcat):
            continue
        records.extend(
            collect_category_records(
                row=row,
                category=subcat,
                path=path + [subcat],
                depth=depth + 1,
                max_depth=max_depth,
                max_files_per_category=max_files_per_category,
                visited_paths=visited_paths,
            )
        )
    return records


def crawl_root_manifest_sample(
    sample_series: list[str],
    max_files_per_category: int = 10,
    max_depth: int = 0,
    db_path: str = IMAGE_DB_PATH,
) -> pd.DataFrame:
    """Crawl manifest records for selected series; max_depth=0 means root only."""
    models = load_commons_models()
    sample = models[models["series"].isin(sample_series)].copy()
    sample = sample[sample["commons_root_category"].notna()]

    conn = init_image_db(db_path)
    all_records = []
    try:
        for _, row in sample.iterrows():
            root_category = row["commons_root_category"]
            records = collect_category_records(
                row=row,
                category=root_category,
                path=[root_category],
                depth=0,
                max_depth=max_depth,
                max_files_per_category=max_files_per_category,
                visited_paths=set(),
            )

            records_df = pd.DataFrame(records)
            if not records_df.empty:
                # Write one category at a time so each category can keep its own source_scope.
                for category, group in records_df.groupby("category", sort=False):
                    first_path = json.loads(group.iloc[0]["category_path_json"])
                    source_scope = "root" if len(first_path) == 1 else "recursive"
                    parent_category = first_path[-2] if len(first_path) > 1 else None
                    # Compatible with older notebook kernels where upsert_image_records
                    # was defined before parent_category existed. Re-run the write helper
                    # cell when possible; this fallback keeps interactive testing moving.
                    try:
                        upsert_image_records(
                            conn,
                            category,
                            group.to_dict("records"),
                            source_scope=source_scope,
                            parent_category=parent_category,
                        )
                    except TypeError as exc:
                        if "parent_category" not in str(exc):
                            raise
                        upsert_image_records(
                            conn,
                            category,
                            group.to_dict("records"),
                            source_scope=source_scope,
                        )
                        if parent_category:
                            conn.execute(
                                "UPDATE categories SET parent_category = COALESCE(parent_category, ?) WHERE category = ?",
                                (parent_category, category),
                            )
                            conn.commit()

            all_records.extend(records)
            print(f'{row["series"]}: Category:{root_category} -> {len(records)} files total')

        purged = purge_non_image_manifest_records(conn)
        print(f'MIME filter removed {purged} non-image rows from the manifest database')
    finally:
        conn.close()

    return pd.DataFrame(all_records)


In [14]:
# max_depth=0 表示只抓 root category；调成 1/2 即可测试递归 subcategory。
sample_manifest = crawl_root_manifest_sample(
    ["E231系", "キハ40系", "EF510形", "DD51形", "C57形", "キハ141系","E233系", "E353系", "E657系", "E5系", "E6系"],
    max_files_per_category=30,
    max_depth=4,
)
sample_manifest[["series", "category", "category_path_json", "file_title", "image_url", "excluded", "exclude_reason"]].head(20)


depth=0 Category:Shinkansen E5 -> 30 files
depth=1 Category:Shinkansen E5 on Hokkaidō Shinkansen -> 0 files
depth=2 Category:Shinkansen E5 at Shin-Aomori Station -> 18 files
depth=2 Category:Shinkansen E5 at Shin-Hakodate-Hokuto Station -> 10 files
depth=1 Category:Shinkansen E5 on Tōhoku Shinkansen -> 23 files
depth=2 Category:Shinkansen E5 at Hachinohe Station -> 2 files
depth=2 Category:Shinkansen E5 at Kōriyama Station (Fukushima) -> 5 files
depth=2 Category:Shinkansen E5 at Morioka Station -> 11 files
depth=2 Category:Shinkansen E5 at Ōmiya Station (Saitama) -> 21 files
depth=2 Category:Shinkansen E5 at Sendai Station (Miyagi) -> 11 files
depth=2 Category:Shinkansen E5 at Shin-Aomori Station -> 18 files
depth=2 Category:Shinkansen E5 at Tokyo Station -> 28 files
E5系: Category:Shinkansen E5 -> 177 files total
depth=0 Category:Shinkansen E6 -> 11 files
depth=1 Category:Shinkansen E6 on Akita Shinkansen -> 20 files
depth=2 Category:Shinkansen E6 at Akita Station -> 15 files
depth=2 C

,series,category,category_path_json,file_title,image_url,excluded,exclude_reason
0,E5系,Shinkansen E5,"[""Shinkansen E5""]",File:E5 All-laps-gangway-bellows.jpg,https://upload.wikimedia.org/wikipedia/commons...,0,NaN
1,E5系,Shinkansen E5,"[""Shinkansen E5""]",File:E5 S11 Sendai 20090725.JPG,https://upload.wikimedia.org/wikipedia/commons...,0,NaN
2,E5系,Shinkansen E5,"[""Shinkansen E5""]","File:E5 Serise Shinkansen , E5系 新幹線 - panorami...",https://upload.wikimedia.org/wikipedia/commons...,0,NaN
3,E5系,Shinkansen E5,"[""Shinkansen E5""]",File:E514-1.jpg,https://upload.wikimedia.org/wikipedia/commons...,0,NaN
4,E5系,Shinkansen E5,"[""Shinkansen E5""]",File:E515-1.jpg,https://upload.wikimedia.org/wikipedia/commons...,0,NaN
5,E5系,Shinkansen E5,"[""Shinkansen E5""]",File:E525-1.jpg,https://upload.wikimedia.org/wikipedia/commons...,0,NaN
6,E5系,Shinkansen E5,"[""Shinkansen E5""]",File:E525-101.jpg,https://upload.wikimedia.org/wikipedia/commons...,0,NaN
7,E5系,Shinkansen E5,"[""Shinkansen E5""]",File:E525-401.jpg,https://upload.wikimedia.org/wikipedia/commons...,0,NaN
8,E5系,Shinkansen E5,"[""Shinkansen E5""]",File:E526-101.jpg,https://upload.wikimedia.org/wikipedia/commons...,0,NaN
9,E5系,Shinkansen E5,"[""Shinkansen E5""]",File:E526-201.jpg,https://upload.wikimedia.org/wikipedia/commons...,0,NaN


In [147]:
sample_manifest = crawl_root_manifest_sample(
    ["キハ40系"],
    max_files_per_category=3,
    max_depth=4,
)
sample_manifest[["series", "category", "category_path_json", "file_title", "image_url", "excluded", "exclude_reason"]].head(50)

depth=0 Category:JNR Kiha 40 series -> 0 files
depth=1 Category:JNR Kiha 40 -> 3 files


KeyboardInterrupt: 

### Kiha 40 category DAG 速览

用少量深度画出 `JNR Kiha 40 series` 的 subcategory 关系，帮助判断递归路径是否符合直觉。

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx


def collect_category_edges(root: str, max_depth: int = 2) -> list[tuple[str, str]]:
    """Collect subcategory edges under a Commons category for quick graph inspection."""
    edges = []
    queue = [(root, 0)]
    expanded = set()

    while queue:
        category, depth = queue.pop(0)
        if category in expanded or depth >= max_depth:
            continue
        expanded.add(category)

        for subcat in fetch_subcategories(category) or []:
            if has_excluded_pattern(subcat, CATEGORY_EXCLUDE_PATTERNS):
                continue
            edges.append((category, subcat))
            queue.append((subcat, depth + 1))

    return edges


root = "JNR Kiha 40 series"
edges = collect_category_edges(root, max_depth=3)
graph = nx.DiGraph(edges)
graph.add_node(root)

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(graph, seed=42, k=1.1)
nx.draw_networkx_edges(graph, pos, arrows=True, alpha=0.35, width=1)
nx.draw_networkx_nodes(graph, pos, node_size=850, node_color="#e7f0ff", edgecolors="#4267a8")
nx.draw_networkx_labels(graph, pos, font_size=8)
plt.title(f"Category DAG sample: {root} (depth<=3, {len(graph.nodes)} nodes, {len(edges)} edges)")
plt.axis("off")
plt.show()


### 测试数据重置

下面的重置代码默认全部注释。确认要清空测试 manifest 和已下载图片后，再手动取消注释运行。


In [ ]:
#⚠️ 手动重置测试数据库/图片目录时再取消注释运行。

# import shutil

# conn = init_image_db()
# conn.executescript("""
#     DELETE FROM image_categories;
#     DELETE FROM images;
#     DELETE FROM categories;
# """)
# conn.commit()
# conn.close()

# shutil.rmtree(os.path.join(PROJECT_ROOT, "data", "img"), ignore_errors=True)
# print("manifest database and data/img reset")


manifest database and data/img reset


### 小样本收集与下载

先用少量车型测试完整流程：抓取 manifest，排除明显内装/部件类文件，然后把原图下载到 `data/img/<series>/`。SQLite 里保存相对路径，方便项目迁移。


In [19]:
from pathlib import Path
from urllib.parse import unquote, urlparse
import time
IMG_ROOT = os.path.join(PROJECT_ROOT, "data", "img")


def safe_path_component(value: str, max_len: int = 120) -> str:
    """Sanitize one path component while keeping readable train/category names."""
    value = unquote(str(value or "")).removeprefix("File:").strip()
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"\s+", " ", value).strip(" .")
    return (value or "unnamed")[:max_len]


def local_image_path(record: pd.Series | dict, img_root: str = IMG_ROOT) -> tuple[str, str]:
    """Return absolute path and DB-relative path for one downloaded image."""
    get = record.get if isinstance(record, dict) else lambda key, default=None: record.get(key, default)
    series_dir = safe_path_component(get("series"))
    file_name = safe_path_component(get("file_title"))
    sha1 = (get("sha1") or "")[:6] #改成了SHA1前6位，也免得太长
    if sha1:
        file_name = f"{sha1}_{file_name}"

    abs_path = os.path.join(img_root, series_dir, file_name)
    # Store paths relative to the SQLite database directory, so data/img becomes img/.
    rel_path = os.path.relpath(abs_path, os.path.dirname(IMAGE_DB_PATH))
    return abs_path, rel_path


def _download_retry_sleep(attempt: int, response: httpx.Response | None = None) -> float:
    """Return a polite retry delay, honoring Retry-After when Commons sends it."""
    if response is not None and response.headers.get("Retry-After"):
        try:
            return min(float(response.headers["Retry-After"]), 30.0)
        except ValueError:
            pass
    return min(1.5 * (2 ** attempt) + random.uniform(0, 0.5), 30.0)


def download_one_image(
    record: pd.Series,
    conn: sqlite3.Connection,
    img_root: str = IMG_ROOT,
    max_retries: int = 3,
) -> dict:
    """Download one manifest image, retry transient failures, and update SQLite."""
    time.sleep(0.3)  # 礼貌性停一下
    if int(record.get("excluded", 0)):
        return {"file_title": record["file_title"], "status": "skipped_excluded", "path": None, "error": None}

    image_url = record.get("image_url")
    error = None
    if not image_url:
        status, rel_path, error = "missing_url", None, "image_url is empty"
    else:
        abs_path, rel_path = local_image_path(record, img_root=img_root)
        os.makedirs(os.path.dirname(abs_path), exist_ok=True)

        if os.path.exists(abs_path) and os.path.getsize(abs_path) > 0:
            status = "downloaded"
        else:
            status = "failed"
            for attempt in range(max_retries + 1):
                try:
                    with httpx.stream("GET", image_url, headers=COMMONS_HEADERS, timeout=60, follow_redirects=True) as resp:
                        resp.raise_for_status()
                        with open(abs_path, "wb") as f:
                            for chunk in resp.iter_bytes():
                                if chunk:
                                    f.write(chunk)
                    status, error = "downloaded", None
                    break

                except httpx.HTTPStatusError as exc:
                    code = exc.response.status_code
                    error = f"HTTP {code}: {exc.response.reason_phrase}"
                    retryable = code in {429, 502, 503, 504}
                    if retryable and attempt < max_retries:
                        sleep_s = _download_retry_sleep(attempt, exc.response)
                        print(f'↻ {record["file_title"]}: {error}, retry in {sleep_s:.1f}s ({attempt + 1}/{max_retries})')
                        time.sleep(sleep_s)
                        continue
                    print(f'✗ {record["file_title"]}: {error}')
                    rel_path = None
                    break

                except httpx.HTTPError as exc:
                    error = f"{type(exc).__name__}: {exc}"
                    if attempt < max_retries:
                        sleep_s = _download_retry_sleep(attempt)
                        print(f'↻ {record["file_title"]}: {error}, retry in {sleep_s:.1f}s ({attempt + 1}/{max_retries})')
                        time.sleep(sleep_s)
                        continue
                    print(f'✗ {record["file_title"]}: {error}')
                    rel_path = None
                    break

    conn.execute(
        """
        UPDATE images
        SET download_status = ?, downloaded_path = ?
        WHERE series = ? AND category = ? AND file_title = ?
        """,
        (status, rel_path, record["series"], record["category"], record["file_title"]),
    )
    return {"file_title": record["file_title"], "status": status, "path": rel_path, "error": error}


def download_manifest_dataframe(
    manifest_df: pd.DataFrame,
    limit: int = -1,
    include_excluded: bool = False,
    db_path: str = IMAGE_DB_PATH,
    img_root: str = IMG_ROOT,
) -> pd.DataFrame:
    """Download images from a manifest DataFrame and persist relative paths in SQLite."""
    if manifest_df.empty:
        return pd.DataFrame(columns=["file_title", "status", "path", "error"])

    work_df = manifest_df.copy()
    if not include_excluded:
        work_df = work_df[work_df["excluded"] == 0]
    work_df = work_df.head(limit) if limit > 0 else work_df

    conn = init_image_db(db_path)
    results = []
    try:
        for _, record in work_df.iterrows():
            results.append(download_one_image(record, conn, img_root=img_root))
        conn.commit()
    finally:
        conn.close()

    return pd.DataFrame(results)


In [20]:

# 小样本：先收集 manifest，再下载少量未排除图片。max_depth 可改为 0/1/2 控制递归。
collection_manifest = crawl_root_manifest_sample(
    ["E231系", "キハ40系", "EF510形", "DD51形", "C57形", "キハ141系","E233系", "E353系", "E657系", "E5系", "E6系"],
    max_files_per_category=7,
    max_depth=4,
)



depth=0 Category:Shinkansen E5 -> 7 files
depth=1 Category:Shinkansen E5 on Hokkaidō Shinkansen -> 0 files
depth=2 Category:Shinkansen E5 at Shin-Aomori Station -> 7 files
depth=2 Category:Shinkansen E5 at Shin-Hakodate-Hokuto Station -> 7 files
depth=1 Category:Shinkansen E5 on Tōhoku Shinkansen -> 7 files
depth=2 Category:Shinkansen E5 at Hachinohe Station -> 2 files
depth=2 Category:Shinkansen E5 at Kōriyama Station (Fukushima) -> 5 files
depth=2 Category:Shinkansen E5 at Morioka Station -> 7 files
depth=2 Category:Shinkansen E5 at Ōmiya Station (Saitama) -> 7 files
depth=2 Category:Shinkansen E5 at Sendai Station (Miyagi) -> 7 files


KeyboardInterrupt: 

In [ ]:
download_result = download_manifest_dataframe(sample_manifest, limit=10000)
download_result

In [ ]:
#TODO: 保存更多图片，总结可以排除的类别和文件名关键词，优化排除规则
#TODO: 对category trace生成unique集合，集合上传LLM进行运营公司/涂装/线路/番台/特殊编成总结，然后回写Metadata。

In [23]:
# Kernel restart 后：从 SQLite manifest 读取未下载图片并继续下载。
# 先运行前面的 import/config、DB helper、download helper cells，再运行本 cell。
def load_pending_manifest_from_db(
    db_path: str = IMAGE_DB_PATH,
    limit: int = -1,
    retry_failed: bool = True,
) -> pd.DataFrame:
    """Load non-excluded manifest rows that still need downloading from SQLite."""
    status_filter = "download_status != 'downloaded'" if retry_failed else "download_status = 'not_started'"
    limit_sql = "" if limit <= 0 else f" LIMIT {int(limit)}"
    sql = f"""
        SELECT *
        FROM images
        WHERE excluded = 0
          AND image_url IS NOT NULL
          AND {status_filter}
        ORDER BY id
        {limit_sql}
    """
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(sql, conn)


pending_manifest = load_pending_manifest_from_db(limit=-1, retry_failed=True)
print(f"pending download rows: {len(pending_manifest)}")

download_result = download_manifest_dataframe(pending_manifest, limit=10000)
if download_result.empty:
    print("nothing to download")
else:
    display(download_result["status"].value_counts())
    display(download_result.head())


pending download rows: 1449
↻ File:美援臺灣火車.jpg: HTTP 429: Your bot is making too many requests. Please reduce your request rate or contact bot-traffic@wikimedia.org (f263c81), retry in 11.0s (1/3)
↻ File:大阪臨港線 浪速駅発貨物列車.JPG: HTTP 429: Your bot is making too many requests. Please reduce your request rate or contact bot-traffic@wikimedia.org (f263c81), retry in 11.0s (1/3)
↻ File:JRE Kuha-E231-39 Outside-LED.jpg: HTTP 429: Your bot is making too many requests. Please reduce your request rate or contact bot-traffic@wikimedia.org (f263c81), retry in 11.0s (1/3)
↻ File:JR East E231 Series Train at Yurakucho Station.JPG: HTTP 429: Your bot is making too many requests. Please reduce your request rate or contact bot-traffic@wikimedia.org (f263c81), retry in 11.0s (1/3)
↻ File:East Japan Railway Company series E233-2000 houkoumakuLED-1 at Abiko station.jpg: HTTP 429: Your bot is making too many requests. Please reduce your request rate or contact bot-traffic@wikimedia.org (f263c81), retry in 11.0

status
downloaded    1449
Name: count, dtype: int64

,file_title,status,path,error
0,File:19th Century Hall SL and Piano Museum 201...,downloaded,img/C57形/b87e35_19th Century Hall SL and Piano...,None
1,File:Aizu Wakamatsu Station-4.JPG,downloaded,img/C57形/0b6c02_Aizu Wakamatsu Station-4.JPG,None
2,File:JNR C57 N Scale train - March 2025.jpg,downloaded,img/C57形/b8c593_JNR C57 N Scale train - March ...,None
3,File:Monuments in front of 19th century hall.jpg,downloaded,img/C57形/1da6bc_Monuments in front of 19th cen...,None
4,File:Wheel of C57 68 in Tsuyama Railroad Educa...,downloaded,img/C57形/3f7c14_Wheel of C57 68 in Tsuyama Rai...,None


In [8]:
import sqlite3
# 随机抽几条 manifest 给本地 LLM 调 prompt。
def sample_manifest_for_llm(
    db_path: str = db_path,
    n: int = 20,
    include_excluded: bool = False,
) -> pd.DataFrame:
    where = "" if include_excluded else "WHERE excluded = 0"
    sql = f"""
        SELECT
            id AS image_id,
            series,
            root_category,
            category,
            category_path_json,
            file_title,
            exclude_reason
        FROM images
        {where}
        ORDER BY random()
        LIMIT {int(n)}
    """
    with sqlite3.connect(db_path) as conn:
        samples = pd.read_sql_query(sql, conn)

    samples["category_path"] = samples["category_path_json"].apply(json.loads)
    return samples.drop(columns=["category_path_json"])


llm_manifest_samples = sample_manifest_for_llm(n=20)
display(llm_manifest_samples)

# 复制这一列里的 dict 给 LLM 最方便。
llm_manifest_samples.to_dict("records")


,image_id,series,root_category,category,file_title,exclude_reason,category_path
0,4463,キハ40系,JNR Kiha 40 series,Kiha 47 (JR West),File:JNR Kiha 47 at Hatabu Station (1616153080...,None,"[JNR Kiha 40 series, JNR Kiha 40 series (JR We..."
1,3101,C57形,C57 steam locomotives,TRA CT273,File:Bogie under the tender of TRA CT273 2010-...,None,"[C57 steam locomotives, TRA CT270, TRA CT273]"
2,2339,キハ100系,JR East KiHa 100 series,JR East KiHa 100 series,File:A commuter train at Morioka Station.jpg,None,[JR East KiHa 100 series]
3,5339,キハ40系,JNR Kiha 40 series,Koshino Shu*Kura,File:越乃Shu*Kura.JPG,None,"[JNR Kiha 40 series, JNR Kiha 40 series (JR Ea..."
4,1307,E5系,Shinkansen E5,Shinkansen E5 at Shin-Aomori Station,"File:E5 Serise Shinkansen , E5系 新幹線 - panorami...",None,"[Shinkansen E5, Shinkansen E5 on Tōhoku Shinka..."
5,5234,キハ40系,JNR Kiha 40 series,Hanayome Noren,File:特急花嫁のれん.JPG,None,"[JNR Kiha 40 series, JNR Kiha 40 series (JR We..."
6,2997,C57形,C57 steam locomotives,C57 139 steam locomotive,"File:JNR Class C57 , 国鉄C57形蒸気機関車 - Panoramio 9...",None,"[C57 steam locomotives, C57 steam locomotives ..."
7,1651,E233系,JR East E233,JR East E233-7000,File:At Tokyo 2024 111.jpg,None,"[JR East E233, JR East E233-7000]"
8,1327,E5系,Shinkansen E5,Shinkansen E5 on Tōhoku Shinkansen,File:E6系S12編成+E5系S11編成試運転.JPG,None,"[Shinkansen E5, Shinkansen E5 on Tōhoku Shinka..."
9,2394,キハ100系,JR East KiHa 100 series,Oykot,File:JREast Kiha110-235 oykot 20150510 02.jpg,None,"[JR East KiHa 100 series, Kiha 110-200, Oykot]"


[{'image_id': 4463,
  'series': 'キハ40系',
  'root_category': 'JNR Kiha 40 series',
  'category': 'Kiha 47 (JR West)',
  'file_title': 'File:JNR Kiha 47 at Hatabu Station (16161530809).jpg',
  'exclude_reason': None,
  'category_path': ['JNR Kiha 40 series',
   'JNR Kiha 40 series (JR West)',
   'Kiha 47 (JR West)']},
 {'image_id': 3101,
  'series': 'C57形',
  'root_category': 'C57 steam locomotives',
  'category': 'TRA CT273',
  'file_title': 'File:Bogie under the tender of TRA CT273 2010-02-02.jpg',
  'exclude_reason': None,
  'category_path': ['C57 steam locomotives', 'TRA CT270', 'TRA CT273']},
 {'image_id': 2339,
  'series': 'キハ100系',
  'root_category': 'JR East KiHa 100 series',
  'category': 'JR East KiHa 100 series',
  'file_title': 'File:A commuter train at Morioka Station.jpg',
  'exclude_reason': None,
  'category_path': ['JR East KiHa 100 series']},
 {'image_id': 5339,
  'series': 'キハ40系',
  'root_category': 'JNR Kiha 40 series',
  'category': 'Koshino Shu*Kura',
  'file_title